# 💬 ذكاء مراجعات العملاء — تصنيف المشاعر ونمذجة المواضيع (Review Intelligence: Sentiment + Topics)
### مشروع C5 — مسار علم البيانات (Data Science Track) ⭐

---
## 🎯 المشكلة التجارية (Business Problem)
شركة بتستقبل آلاف المراجعات يومياً ومش قادرة تقراها كلها. عايزين نظام:
1. **يصنّف مشاعر كل مراجعة** تلقائياً (إيجابي / سلبي / محايد) — عشان يرصدوا المشاكل بسرعة.
2. **يكتشف المواضيع المتكررة** (Topics) في المراجعات — عشان يعرفوا الناس بتشتكي/تمدح في إيه.

**نوعا المشكلة:** تصنيف نصوص (Text Classification) + نمذجة مواضيع غير موجّهة (Topic Modeling).

## 📦 ما الذي يثبته المشروع
تنظيف النصوص · TF-IDF · مقارنة مصنّفات نصوص · التعامل مع عدم التوازن ·
تمثيلات دلالية (LSA Embeddings) · **نمذجة المواضيع (LDA)** · التقييم بمصفوفة اللخبطة.


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "data_science/c5_nlp_reviews"          # مسار المشروع داخل portfolio/
PKGS    = []          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| تنظيف النص (lowercasing, stopwords, tokenization) | Jurafsky — *Speech & Language Processing* (ch.2) | كل مهمة NLP بتبدأ بالتنظيف |
| **TF-IDF** | Jurafsky (ch.6) / Géron | تحويل النص لأرقام يفهمها الموديل |
| Naive Bayes للنصوص | Jurafsky (ch.4) | خط الأساس الكلاسيكي لتصنيف النص |
| Linear SVM / Logistic للنصوص | ISLR (ch.9) / Géron | الأقوى عادةً مع TF-IDF |
| عدم التوازن (class_weight, macro-F1) | Géron (ch.3) | المراجعات السلبية قليلة لكنها الأهم |
| تمثيلات دلالية (LSA / Embeddings) | Jurafsky (ch.6) | فهم المعنى لا الكلمات فقط |
| **نمذجة المواضيع (LDA)** | Blei et al. / Jurafsky | اكتشاف الثيمات المخفية بدون تسميات |

> 🎯 **بيُستخدم في الواقع:** مراقبة سمعة العلامة التجارية، تحليل تذاكر الدعم، تصنيف بريد، تحليل السوشيال ميديا.
> 🛠️ **ملاحظة إنتاجية:** للنصوص الحقيقية المعقّدة تُستخدم Embeddings من `sentence-transformers` أو BERT — هنا نستخدم LSA (خفيف ويشتغل بدون GPU).


## 0️⃣ المكتبات

In [ ]:
import numpy as np, pandas as pd, re
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid'); np.random.seed(42)
print('ready ✓')

## 1️⃣ تحميل واستكشاف البيانات (EDA)

In [ ]:
df = pd.read_csv('data/product_reviews_data.csv')
# TODO: اطبع shape وتوزيع sentiment، وارسم توزيع المشاعر وطول النص
print(df['sentiment'].value_counts())
df.head(3)

> 📌 لاحظ: المراجعات الإيجابية أكتر بكتير → **بيانات غير متوازنة**. لذلك هنستخدم `macro-F1` و `class_weight`.

## 2️⃣ تنظيف النصوص (Text Preprocessing)

In [ ]:
STOP = set('the a an and or but is are was were be to of in on for with this that it'.split())
def clean(t):
    # TODO: lowercase + شيل غير الحروف + شيل stopwords والكلمات القصيرة
    ...
df['clean'] = df['review_text'].apply(clean)
print(df['clean'].iloc[0])

## 3️⃣ تحويل TF-IDF + تقسيم البيانات

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
# TODO: قسّم البيانات (stratify) ثم درّب TfidfVectorizer مع ngram_range=(1,2)
X_train, X_test, y_train, y_test = ...
tfidf = ...
Xtr, Xte = ...
print('shape:', Xtr.shape)

## 4️⃣ مقارنة مصنّفات النصوص (Model Battle)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report, confusion_matrix
# TODO: درّب وقارن الـ 3 موديلات بمقياس macro-F1 واختر الأفضل
models = {...}
# ...
best_model = ...

## 5️⃣ تقييم أفضل موديل

In [ ]:
# TODO: classification_report + confusion matrix لأفضل موديل
pred = best_model.predict(Xte)
print(classification_report(y_test, pred))

## 6️⃣ التمثيلات الدلالية (LSA Embeddings) + تصوير
نختزل آلاف أبعاد TF-IDF لبُعدين باستخدام **Truncated SVD (LSA)** ونرسم المراجعات ملوّنة بالمشاعر.

In [ ]:
from sklearn.decomposition import TruncatedSVD
# TODO: TruncatedSVD(n_components=2) على مصفوفة TF-IDF كاملة، ثم ارسمها ملوّنة بالمشاعر
svd = ...
emb = ...

## 7️⃣ نمذجة المواضيع (Topic Modeling — LDA) 🔍
LDA بيكتشف "مواضيع" مخفية بدون أي تسميات. كل موضوع = مجموعة كلمات بتظهر مع بعض.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
# TODO: CountVectorizer ثم LDA بـ 4 مواضيع، واطبع أهم 8 كلمات لكل موضوع
cv = ...
lda = ...

## 8️⃣ الخلاصة والتوصيات (Conclusion)
- **التصنيف:** LinearSVC/LogReg مع TF-IDF حقّقوا أعلى macro-F1 — يقدروا يصنّفوا المراجعات تلقائياً.
- **عدم التوازن:** استخدمنا `class_weight='balanced'` و `macro-F1` عشان الفئة السلبية المهمة ما تضيعش.
- **المواضيع (LDA):** كشفت ثيمات زي الجودة، التوصيل، السعر — معلومة قيّمة لفريق المنتج.
- **التوصية:** أتمتة تصنيف المراجعات الواردة + لوحة متابعة للمواضيع السلبية الصاعدة.
- **الخطوة القادمة:** استبدال TF-IDF بـ embeddings من `sentence-transformers` لرفع الدقة على النصوص المعقّدة.

> ✅ **اللي اتعلمته:** تنظيف النص، TF-IDF، مقارنة مصنّفات، عدم التوازن، LSA، و LDA.
